In [1]:
import os
import numpy as np
import pandas as pd
from glob import glob
from scipy.signal import butter, filtfilt
from scipy.stats import iqr
from scipy.fftpack import fft

## Import Data

In [2]:
df = pd.read_csv("dataset/derivative_mid_5min/sub-001.csv")

In [3]:
participants_file = "dataset/participants.tsv"
participants_df = pd.read_csv(participants_file, sep='\t')
participants_df = participants_df.drop(columns=['Unnamed: 0'])
label_dict = {"A": 0, "F": 1, "C": 2}
participants_df["Group"] = participants_df["Group"].map(label_dict)

participants_df.head()

,participant_id,Gender,Age,Group,MMSE,Set
0,sub-023,M,60,0,16,Train
1,sub-021,M,79,0,22,Train
2,sub-003,M,70,0,14,Train
3,sub-020,M,71,0,4,Train
4,sub-012,M,63,0,18,Train


## Define Frequency Bands And Sampling Rate

In [4]:
# Sampling rate and window parameters
fs = 500  # Sampling rate 250Hz
window_size = 5 * fs  # 5-second window
step_size = 2.5 * fs  # 2.5-second overlap

# EEG frequency bands definition
freq_bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 12),
    "Beta": (12, 25),
    "Gamma": (25, 45)
}

# Design a band-pass filter
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

# Compute power spectral density
def band_power(data, fs, band):
    freqs = np.fft.fftfreq(len(data), d=1/fs)
    fft_values = np.abs(fft(data)) ** 2
    band_idx = np.where((freqs >= band[0]) & (freqs <= band[1]))[0]
    # return np.sum(fft_values[band_idx])
    return np.mean(fft_values[band_idx])

## Extract Feature

In [5]:
# Read all EEG files
eeg_files = glob("dataset/derivative_mid_5min/*.csv")
all_data = []

for eeg_file in eeg_files:
    # Get participant ID
    participant_id = os.path.basename(eeg_file).split('.')[0]

    # Find participant information
    participant_info = participants_df[participants_df["participant_id"] == participant_id]
    if participant_info.empty:
        print(f"Skipping {participant_id}: No participant info found.")
        continue
    label = participant_info.iloc[0]["Group"]
    train_test = participant_info.iloc[0]['Set']

    # Read EEG data
    df = pd.read_csv(eeg_file)
    print(f"Processing {eeg_file}")
    channels = df.columns.tolist() 
    num_samples = df.shape[0]

    # Sliding window processing
    feature_data = []
    for start in range(0, num_samples - window_size, int(step_size)):
        window_data = df.iloc[start:start+window_size]
        feature_row = {}

        for ch in channels:
            # signal = window_data[ch].values
            signal = bandpass_filter(window_data[ch].values, 0.5, 48, fs)

            # Compute time-domain features
            feature_row[f"{ch}_Mean"] = np.mean(signal)
            feature_row[f"{ch}_Var"] = np.var(signal)
            feature_row[f"{ch}_IQR"] = iqr(signal)

            # Compute frequency-domain features
            for band in freq_bands:
                feature_row[f"{ch}_{band}"] = band_power(signal, fs, freq_bands[band])
        
        # Add participant information
        feature_row["Label"] = label
        feature_row["train_test"] = train_test
        feature_data.append(feature_row)
    
    feature_df = pd.DataFrame(feature_data)
    all_data.append(feature_df)
    print(f"Finished {participant_id}: {feature_df.shape[0]} rows collected")

Processing dataset/derivative_mid_5min\sub-001.csv
Finished sub-001: 118 rows collected
Processing dataset/derivative_mid_5min\sub-002.csv
Finished sub-002: 118 rows collected
Processing dataset/derivative_mid_5min\sub-003.csv
Finished sub-003: 118 rows collected
Processing dataset/derivative_mid_5min\sub-004.csv
Finished sub-004: 118 rows collected
Processing dataset/derivative_mid_5min\sub-005.csv
Finished sub-005: 118 rows collected
Processing dataset/derivative_mid_5min\sub-006.csv
Finished sub-006: 118 rows collected
Processing dataset/derivative_mid_5min\sub-007.csv
Finished sub-007: 118 rows collected
Processing dataset/derivative_mid_5min\sub-008.csv
Finished sub-008: 118 rows collected
Processing dataset/derivative_mid_5min\sub-009.csv
Finished sub-009: 118 rows collected
Processing dataset/derivative_mid_5min\sub-010.csv
Finished sub-010: 118 rows collected
Processing dataset/derivative_mid_5min\sub-011.csv
Finished sub-011: 118 rows collected
Processing dataset/derivative_mi

## Export Data

In [6]:
# Merge all data
final_df = pd.concat(all_data, ignore_index=True)

final_df.to_csv("dataset/eeg_features.csv", index=False)

In [7]:
final_df.columns

Index(['Fp1_Mean', 'Fp1_Var', 'Fp1_IQR', 'Fp1_Delta', 'Fp1_Theta', 'Fp1_Alpha',
       'Fp1_Beta', 'Fp1_Gamma', 'Fp2_Mean', 'Fp2_Var',
       ...
       'Pz_Mean', 'Pz_Var', 'Pz_IQR', 'Pz_Delta', 'Pz_Theta', 'Pz_Alpha',
       'Pz_Beta', 'Pz_Gamma', 'Label', 'train_test'],
      dtype='object', length=154)